# Checkpoint 06 — cliente completo

Este checkpoint é uma **evolução direta do Checkpoint 05**.

A ideia é manter a mesma estrutura já conhecida:

```text
Firebase → SSE → validar_pacote() → buffers → gráfico
```

e acrescentar somente o que falta para um cliente mais completo:

- os dois canais de PPG;
- detecção de pacotes duplicados e perdidos;
- métricas simples da rede;
- reconexão do SSE;
- registro das amostras;
- exportação para CSV.

As funções `firebase_node_url()`, `ler_pacote_atual()`, `iterar_sse()`,
`pacotes_firebase()`, `validar_pacote()` e `media_movel_segmentos()` são
mantidas com a mesma lógica dos checkpoints anteriores.


## 1. Configuração

Use a mesma URL do Realtime Database utilizada nos checkpoints anteriores.


In [ ]:
# CONFIGURAÇÃO DA OFICINA
FIREBASE_URL = "https://oficina-python-saude-teste-default-rtdb.firebaseio.com"  # Endereço base do Realtime Database usado na oficina.
STREAM_PATH = "oficina/live"  # Nó que contém sempre o pacote mais recente.

DURACAO_SEGUNDOS = 100  # Tempo total de execução deste checkpoint.
JANELA_S = 8             # Duração, em segundos, da janela mostrada no gráfico.
PLOT_A_CADA_S = 0.8      # Intervalo mínimo entre duas atualizações consecutivas da figura.


## 2. Funções já usadas no Checkpoint 05

O bloco abaixo preserva a estrutura já conhecida. Antes de acrescentar funcionalidades,
continuamos recebendo e validando os pacotes exatamente da mesma forma.


In [ ]:
import csv                          # Grava as amostras recebidas em um arquivo CSV.
import json                         # Converte as mensagens JSON recebidas pelo SSE.
import statistics                   # Calcula medianas dos intervalos e latências de rede.
import time                         # Mede duração, intervalos de chegada e controla reconexões.
from collections import deque       # Mantém buffers deslizantes dos sinais.

import matplotlib.pyplot as plt     # Mostra ECG e PPG em tempo real.
import numpy as np                  # Manipula vetores e representa lacunas com np.nan.
import requests                     # Recebe o fluxo do Firebase por HTTP/SSE.
from IPython.display import clear_output, display  # Atualiza o gráfico no mesmo espaço de saída do Colab.


# Função já utilizada anteriormente: monta a URL REST completa do nó do Firebase.
def firebase_node_url(firebase_url, node=STREAM_PATH):
    base = firebase_url.rstrip("/")  # Remove uma barra final para evitar "//" ao montar a URL.
    node = node.strip("/")           # Remove barras no início/fim do caminho antes de juntá-lo à URL.
    return f"{base}/{node}.json"     # O sufixo .json permite acessar o nó pela API REST do Firebase.


# Função já utilizada anteriormente: faz uma leitura única do pacote atual.
def ler_pacote_atual(firebase_url=FIREBASE_URL, node=STREAM_PATH):
    r = requests.get(firebase_node_url(firebase_url, node), timeout=10)  # Faz uma leitura HTTP comum.
    r.raise_for_status()  # Interrompe se o Firebase responder com um código HTTP de erro.
    return r.json()       # Converte a resposta JSON para um objeto Python.


# Função já utilizada anteriormente: separa o fluxo SSE em eventos completos.
def iterar_sse(response):
    """Interpreta os eventos Server-Sent Events retornados pelo Firebase REST."""
    evento = None
    dados = []

    for raw in response.iter_lines(decode_unicode=True):  # Lê continuamente a resposta HTTP, linha a linha.
        if raw is None:
            continue

        linha = raw.strip()  # Remove espaços e quebras de linha antes de interpretar o campo SSE.

        if linha == "":  # Uma linha vazia indica o fim de um evento SSE.
            if evento is not None and dados:
                yield evento, "\n".join(dados)
            evento = None
            dados = []
            continue

        if linha.startswith(":"):  # Linhas iniciadas por ":" são comentários/keep-alive.
            continue

        if linha.startswith("event:"):
            evento = linha.split(":", 1)[1].strip()  # Extrai o tipo do evento.
        elif linha.startswith("data:"):
            dados.append(linha.split(":", 1)[1].lstrip())  # Guarda o conteúdo JSON do evento.


# Função já utilizada anteriormente: mantém uma conexão SSE aberta e entrega os pacotes recebidos.
def pacotes_firebase(firebase_url=FIREBASE_URL, node=STREAM_PATH, duracao_s=20):
    """Gera pacotes recebidos pelo stream SSE do Firebase."""
    url = firebase_node_url(firebase_url, node)
    headers = {"Accept": "text/event-stream"}  # Solicita ao Firebase um fluxo Server-Sent Events.
    inicio = time.time()

    with requests.get(
        url,
        headers=headers,
        stream=True,              # Mantém a conexão aberta para receber vários eventos.
        allow_redirects=True,     # Aceita redirecionamentos do serviço Firebase.
        timeout=(10, 35),         # 10 s para conectar e até 35 s sem dados antes de timeout.
    ) as response:
        response.raise_for_status()

        for evento, texto in iterar_sse(response):
            if time.time() - inicio >= duracao_s:
                return

            if evento in ("cancel", "auth_revoked"):
                raise RuntimeError(f"Firebase encerrou o stream: {evento}")

            if evento not in ("put", "patch"):
                continue

            envelope = json.loads(texto)  # Converte o texto JSON do evento em um dicionário Python.
            data = envelope.get("data")

            # Para esta oficina, cada publicação substitui /oficina/live por um pacote completo.
            if isinstance(data, dict) and "seq" in data:
                yield data


# Função já utilizada anteriormente: confere os campos e converte os sinais para NumPy.
def validar_pacote(pacote):
    campos = (
        "session_id", "seq", "sample_start", "fs", "n",
        "ecg", "ppg_ir", "ppg_red"
    )

    faltando = [k for k in campos if k not in pacote]  # Identifica campos obrigatórios que não chegaram.
    if faltando:
        raise ValueError(f"campos ausentes: {faltando}")

    n = int(pacote["n"])
    ecg = np.asarray(pacote["ecg"], dtype=float)
    ir = np.asarray(pacote["ppg_ir"], dtype=float)
    red = np.asarray(pacote["ppg_red"], dtype=float)

    if not (len(ecg) == len(ir) == len(red) == n):
        raise ValueError("vetores com tamanhos incompatíveis")

    return {
        "session_id": str(pacote["session_id"]),
        "seq": int(pacote["seq"]),
        "sample_start": int(pacote["sample_start"]),
        "fs": float(pacote["fs"]),
        "n": n,
        "timestamp_pub": float(pacote.get("timestamp_pub", 0.0)),
        "ecg": ecg,
        "ppg_ir": ir,
        "ppg_red": red,
    }


# Função já utilizada no Checkpoint 05: média móvel sem atravessar lacunas NaN.
def media_movel_segmentos(x, n=7):
    """Aplica média móvel somente em trechos contínuos que possuem valores válidos."""
    x = np.asarray(x, dtype=float)
    y = np.full_like(x, np.nan)  # Começa a saída com NaN para preservar eventuais lacunas.
    finite = np.isfinite(x)      # Marca as posições que contêm números válidos.

    i = 0
    while i < len(x):
        if not finite[i]:
            i += 1
            continue

        j = i
        while j < len(x) and finite[j]:
            j += 1

        seg = x[i:j]  # Extrai um trecho contínuo sem NaN.

        if len(seg) >= n:
            y[i:j] = np.convolve(seg, np.ones(n) / n, mode="same")
        else:
            y[i:j] = seg

        i = j

    return y


## 3. Novidades do Checkpoint 06

Agora generalizamos o `append_ecg()` do checkpoint anterior para `append_sinais()`:
a mesma lógica de sessão, continuidade temporal e lacunas passa a ser aplicada
simultaneamente a ECG, PPG IR e PPG vermelho.

Também acrescentamos métricas de rede e o registro das amostras para CSV.


In [ ]:
# Buffers de visualização. A lógica é a mesma do buffer de ECG do Checkpoint 05.
sample_buf = deque()
ecg_buf = deque()
ir_buf = deque()
red_buf = deque()

# Lista que conservará todas as linhas da sessão atual para posterior exportação em CSV.
registros = []

# Estado da sessão e da continuidade dos pacotes.
active_session = None
last_sample_end = None
last_seq = None
last_plot = 0.0
ultimo_recebimento = None

# Contadores e métricas.
pacotes = 0
pacotes_ausentes = 0
amostras_ausentes = 0
pacotes_duplicados = 0
sessoes_detectadas = 0
intervalos_ms = []
latencias_ms = []


# Reinicia buffers e contadores quando o publicador inicia uma nova sessão.
def reiniciar_sessao(session_id):
    global active_session, last_sample_end, last_seq, last_plot, ultimo_recebimento
    global pacotes, pacotes_ausentes, amostras_ausentes, pacotes_duplicados
    global sessoes_detectadas

    active_session = session_id
    last_sample_end = None
    last_seq = None
    last_plot = 0.0
    ultimo_recebimento = None

    pacotes = 0
    pacotes_ausentes = 0
    amostras_ausentes = 0
    pacotes_duplicados = 0
    sessoes_detectadas += 1

    intervalos_ms.clear()
    latencias_ms.clear()

    sample_buf.clear()
    ecg_buf.clear()
    ir_buf.clear()
    red_buf.clear()
    registros.clear()

    print(f"Nova sessão detectada: {session_id}")


# Insere NaN nas posições correspondentes às amostras que não chegaram.
# Essa é a mesma ideia usada anteriormente para preservar corretamente o eixo temporal.
def inserir_gap(esperado, sample_start, fs):
    global amostras_ausentes

    gap = sample_start - esperado  # Calcula quantas amostras ficaram ausentes.
    amostras_ausentes += gap

    for sample_idx in range(esperado, sample_start):
        sample_buf.append(sample_idx)
        ecg_buf.append(np.nan)
        ir_buf.append(np.nan)
        red_buf.append(np.nan)

        # Registra também a lacuna no CSV; não inventamos valores para as amostras perdidas.
        registros.append([
            active_session,
            sample_idx,
            sample_idx / fs,
            "",
            "",
            np.nan,
            np.nan,
            np.nan,
            "MISSING",
        ])


# Evolução do append_ecg() do Checkpoint 05:
# acrescenta os três sinais, detecta duplicações/perdas e mantém a janela temporal.
def append_sinais(p):
    global last_sample_end, last_seq, pacotes, pacotes_ausentes, pacotes_duplicados

    if active_session != p["session_id"]:
        reiniciar_sessao(p["session_id"])

    sample_start = p["sample_start"]
    n = p["n"]
    fs = p["fs"]
    seq = p["seq"]

    # Se o pacote começa em uma amostra já recebida, ele é duplicado ou antigo.
    if last_sample_end is not None and sample_start <= last_sample_end:
        pacotes_duplicados += 1
        return False

    # O seq também permite inferir quantos pacotes inteiros deixaram de chegar.
    if last_seq is not None and seq > last_seq + 1:
        pacotes_ausentes += seq - last_seq - 1

    # Mantém a continuidade temporal inserindo NaN onde existirem amostras faltantes.
    if last_sample_end is not None:
        esperado = last_sample_end + 1
        if sample_start > esperado:
            inserir_gap(esperado, sample_start, fs)

    # Acrescenta ao buffer todas as amostras válidas deste pacote.
    for i in range(n):
        sample_idx = sample_start + i
        signal_time = sample_idx / fs

        ecg_i = float(p["ecg"][i])
        ir_i = float(p["ppg_ir"][i])
        red_i = float(p["ppg_red"][i])

        sample_buf.append(sample_idx)
        ecg_buf.append(ecg_i)
        ir_buf.append(ir_i)
        red_buf.append(red_i)

        registros.append([
            active_session,
            sample_idx,
            signal_time,
            seq,
            i,
            ecg_i,
            ir_i,
            red_i,
            "OK",
        ])

    last_seq = seq
    last_sample_end = sample_start + n - 1
    pacotes += 1

    # Conserva no buffer de visualização apenas os últimos JANELA_S segundos.
    minimo = last_sample_end - int(JANELA_S * fs)
    while sample_buf and sample_buf[0] < minimo:
        sample_buf.popleft()
        ecg_buf.popleft()
        ir_buf.popleft()
        red_buf.popleft()

    return True


# Atualiza métricas relacionadas à chegada pela rede.
# Elas são analisadas separadamente do tempo do sinal, que continua vindo de sample_start/fs.
def atualizar_metricas_rede(p, agora):
    global ultimo_recebimento

    if ultimo_recebimento is not None:
        intervalos_ms.append((agora - ultimo_recebimento) * 1000)

    ultimo_recebimento = agora

    if p["timestamp_pub"] > 0:
        latencias_ms.append((agora - p["timestamp_pub"]) * 1000)


# Mostra ECG e PPG mantendo o mesmo princípio temporal dos checkpoints anteriores.
def mostrar_graficos(p, inicio):
    latest = sample_buf[-1]
    tx = np.asarray([(s - latest) / p["fs"] for s in sample_buf], dtype=float)

    bruto = np.asarray(ecg_buf, dtype=float)
    suavizado = media_movel_segmentos(bruto, n=7)  # Reutiliza o processamento introduzido no Checkpoint 05.

    clear_output(wait=True)

    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

    ax[0].plot(tx, bruto, alpha=0.45, label="ECG bruto")
    ax[0].plot(tx, suavizado, linewidth=1.6, label="ECG suavizado")
    ax[0].set_title("ECG — tempo definido pela aquisição")
    ax[0].set_ylabel("amplitude")
    ax[0].grid(True, alpha=0.25)
    ax[0].legend()

    ax[1].plot(tx, list(ir_buf), label="PPG IR")
    ax[1].plot(tx, list(red_buf), label="PPG Red")
    ax[1].set_title("PPG — tempo definido pela aquisição")
    ax[1].set_xlabel("tempo relativo ao sinal (s)")
    ax[1].set_ylabel("contagens")
    ax[1].grid(True, alpha=0.25)
    ax[1].legend()

    fig.tight_layout()
    display(fig)
    plt.close(fig)

    med_intervalo = statistics.median(intervalos_ms[-30:]) if intervalos_ms else float("nan")
    med_lat = statistics.median(latencias_ms[-30:]) if latencias_ms else float("nan")

    print(f"sessão:                     {active_session[:12]}...")
    print(f"tempo de execução:          {time.time() - inicio:7.1f} s")
    print(f"pacotes recebidos:          {pacotes}")
    print(f"pacotes ausentes*:          {pacotes_ausentes}")
    print(f"amostras ausentes*:         {amostras_ausentes}")
    print(f"pacotes duplicados/antigos: {pacotes_duplicados}")
    print(f"intervalo de rede mediano:  {med_intervalo:7.1f} ms")
    print(f"latência aproximada**:      {med_lat:7.1f} ms")
    print("* Inferidos por seq/sample_start.")
    print("** Métrica de transporte; não é usada no eixo temporal do sinal.")


# Mantém a lógica de pacotes_firebase(), mas tenta reconectar se a conexão SSE cair.
def pacotes_firebase_resiliente(firebase_url=FIREBASE_URL, node=STREAM_PATH, duracao_s=100):
    inicio = time.time()

    while time.time() - inicio < duracao_s:
        restante = duracao_s - (time.time() - inicio)

        try:
            # Reutiliza pacotes_firebase(); a única novidade aqui é tentar novamente se a conexão terminar.
            for pacote in pacotes_firebase(
                firebase_url=firebase_url,
                node=node,
                duracao_s=restante,
            ):
                yield pacote

        except KeyboardInterrupt:
            return

        except Exception as exc:
            print(f"Stream interrompido ({type(exc).__name__}: {exc})")

        # Se ainda houver tempo de execução, espera 1 s antes de tentar uma nova conexão.
        if time.time() - inicio < duracao_s:
            print("Tentando reconectar em 1 s...")
            time.sleep(1)


# Grava em CSV os dados acumulados na sessão atual.
def salvar_csv(registros, arquivo="dados_recebidos_oficina.csv"):
    if not registros:
        print("Nenhum dado válido foi recebido; CSV não foi criado.")
        return None

    with open(arquivo, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "session_id",
            "sample_index",
            "signal_time_s",
            "seq",
            "amostra_no_pacote",
            "ecg",
            "ppg_ir",
            "ppg_red",
            "status",
        ])
        w.writerows(registros)

    print(f"Arquivo criado: {arquivo}")
    return arquivo


## 4. Executar o cliente completo

Execute esta célula **antes de iniciar uma nova sessão do publicador**.

Se quiser exercitar perda de pacotes, altere o gerador para `DROPOUT`.
Para uma captura normal, use `NORMAL`.


In [ ]:
inicio = time.time()

for pacote in pacotes_firebase_resiliente(
    firebase_url=FIREBASE_URL,
    node=STREAM_PATH,
    duracao_s=DURACAO_SEGUNDOS,
):
    try:
        p = validar_pacote(pacote)  # Reutiliza exatamente a validação dos checkpoints anteriores.
    except ValueError as exc:
        print("Pacote ignorado:", exc)
        continue

    agora = time.time()

    if not append_sinais(p):  # Pacotes duplicados/antigos não seguem para as etapas seguintes.
        continue

    atualizar_metricas_rede(p, agora)

    # Atualiza a tela em intervalos regulares para não redesenhar a figura a cada pacote.
    if agora - last_plot >= PLOT_A_CADA_S and len(sample_buf) > 5:
        last_plot = agora
        mostrar_graficos(p, inicio)

clear_output(wait=True)
print("Execução encerrada.")
print(f"Sessões detectadas:              {sessoes_detectadas}")
print(f"Pacotes recebidos na sessão:     {pacotes}")
print(f"Pacotes ausentes inferidos:      {pacotes_ausentes}")
print(f"Amostras ausentes inferidas:     {amostras_ausentes}")
print(f"Pacotes duplicados/antigos:      {pacotes_duplicados}")

if intervalos_ms:
    print(f"Intervalo mediano da rede:       {statistics.median(intervalos_ms):.1f} ms")
    print(f"Maior intervalo observado:       {max(intervalos_ms):.1f} ms")

if latencias_ms:
    print(f"Latência mediana aproximada:     {statistics.median(latencias_ms):.1f} ms")

arquivo_csv = salvar_csv(registros)


## 5. Opcional — baixar o CSV

O CSV contém o índice absoluto da amostra (`sample_index`), o tempo reconstruído
do sinal (`signal_time_s`) e linhas `MISSING` para as lacunas detectadas.


In [ ]:
if arquivo_csv:
    from google.colab import files  # Recurso do Colab para transferir um arquivo para o computador.
    files.download(arquivo_csv)


## 6. Pós-processamento do CSV com Pandas

Aqui `pandas` passa a ser útil porque agora temos um arquivo tabular completo.


In [ ]:
import pandas as pd  # Carrega e manipula o CSV como uma tabela (DataFrame).

if arquivo_csv:
    df = pd.read_csv(arquivo_csv)
    display(df.head())

    print("amostras/linhas:", len(df))
    print(df["status"].value_counts(dropna=False))
